In [2]:
# 1. Rileva se il notebook è in esecuzione su Google Colab
import os
import sys
import subprocess

# 1. Rileva se il notebook è in esecuzione su Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 2. Esegui le istruzioni condizionali solo su Colab
if IN_COLAB:
    print("Google Colab rilevato: configurazione dell'ambiente in corso...")
    
    # Installazione di skrl
    subprocess.run(["pip", "install", "-q", "skrl"], check=True)
    
    # Clonazione o aggiornamento del repository DetectorOptimizer
    repo_name = "DetectorOptimizer"
    repo_url = "https://github.com/Stelfano/DetectorOptimizer.git"
    
    if not os.path.exists(repo_name):
        print(f"Clonazione del repository {repo_url}...")
        subprocess.run(["git", "clone", repo_url], check=True)
    else:
        print(f"La cartella '{repo_name}' esiste già. Aggiornamento in corso...")
        subprocess.run(["git", "-C", repo_name, "pull"], check=True)
        
    # Aggiunge la cartella del repository al Python path per poter importare i moduli
    repo_path = os.path.abspath(repo_name) if 'os.path-abspath' not in globals() else os.path.abspath(repo_name)
    # Nota: usiamo os.path.abspath standard
    repo_path = os.path.abspath(repo_name)
    if repo_path not in sys.path:
        sys.path.append(repo_path)
        
    # Installazione automatica delle dipendenze del repository (se presenti)
    requirements_path = os.path.join(repo_name, "requirements.txt")
    if os.path.exists(requirements_path):
        print("Installazione delle dipendenze da requirements.txt...")
        subprocess.run(["pip", "install", "-q", "-r", requirements_path], check=True)
        
    # Avvio di TensorBoard tramite le magie di IPython
    try:
        get_ipython().run_line_magic("load_ext", "tensorboard")
        get_ipython().run_line_magic("tensorboard", "--logdir runs")
        print("TensorBoard avviato con successo.")
    except Exception as e:
        print(f"Impossibile avviare automaticamente TensorBoard: {e}")
else:
    print("Ambiente locale rilevato: salto il download del repository e l'avvio di TensorBoard.")

Google Colab rilevato: configurazione dell'ambiente in corso...
La cartella 'DetectorOptimizer' esiste già. Aggiornamento in corso...


Reusing TensorBoard on port 6006 (pid 843), started 1:03:35 ago. (Use '!kill 843' to kill it.)

<IPython.core.display.Javascript object>

TensorBoard avviato con successo.


In [3]:
from include.envs import SimpleRoTDetector

env = SimpleRoTDetector(n_tracks=100, spatial_res=0.1, time_res=0.3, tof_margin=50.0, debug=False)

In [4]:
from skrl.models.torch import Model, MultivariateGaussianMixin, DeterministicMixin
from torch import nn
import torch


class Actor(MultivariateGaussianMixin, Model):
    def __init__(self, observation_space, action_space, device):
        Model.__init__(self, observation_space=observation_space, action_space=action_space, device=device)
        MultivariateGaussianMixin.__init__(self, clip_actions=True, min_log_std=-20, max_log_std=2)

        self.net = nn.Sequential(
            nn.Linear(self.num_observations, 2**10), nn.Sigmoid(),
            nn.Linear(2**10, 2**12), nn.Sigmoid(),
            nn.Linear(2**12, 2**10), nn.Sigmoid(),
            nn.Linear(2**10, self.num_actions) 
        )

        self.log_std_parameter = nn.Parameter(torch.zeros(self.num_actions))

    def compute(self, inputs, role=""):
        return self.net(inputs["observations"]), {"log_std": self.log_std_parameter}


class Critic(DeterministicMixin, Model):
    def __init__(self, observation_space, action_space, device):
        Model.__init__(self, observation_space=observation_space, action_space=action_space, device=device)
        DeterministicMixin.__init__(self, clip_actions=False)

        self.net = nn.Sequential(
            nn.Linear(self.num_observations+self.num_actions, 2**12), nn.Sigmoid(),
            nn.Linear(2**12, 2**10), nn.Sigmoid(),
            nn.Linear(2**10, 2**8), nn.Sigmoid(),
            nn.Linear(2**8, 1),
        )

    def compute(self, inputs, role=""):
        x = torch.cat([inputs["observations"], inputs["taken_actions"]], dim=-1)
        return self.net(x), {}


In [5]:
torch.autograd.set_detect_anomaly(True)

In [6]:
from skrl.envs.wrappers.torch import wrap_env
from skrl.memories.torch import RandomMemory
from skrl.resources.noises.torch import GaussianNoise
from skrl.agents.torch.sac import SAC, SAC_CFG
from skrl.resources.preprocessors.torch import RunningStandardScaler

device = "cuda" if torch.cuda.is_available() else "cpu"

env = wrap_env(env, wrapper = 'gymnasium')
memory = RandomMemory(memory_size=20000, num_envs=env.num_envs, device=device)

models = {
    "policy": Actor(env.observation_space, env.action_space, device),
    "critic_1": Critic(env.observation_space, env.action_space, device),
    "critic_2": Critic(env.observation_space, env.action_space, device),
    "target_critic_1": Critic(env.observation_space, env.action_space, device),
    "target_critic_2": Critic(env.observation_space, env.action_space, device),
}

cfg = SAC_CFG(
    batch_size = 16,
    discount_factor=0.98,
    learning_rate=1e-4,
    random_timesteps=1000,
    learning_starts=1000,
    gradient_steps=1
)
cfg.experiment.write_interval = 20
cfg.experiment.checkpoint_interval = 0
cfg.experiment.experiment_name = "SAC-RoT"
cfg.experiment.directory = "./runs"
cfg.close_environment_at_exit = True
cfg.timesteps = 50000
cfg.disable_progressbar = False
cfg.headless = True
cfg.environment_info = True
cfg.state_preprocessor = RunningStandardScaler
cfg.state_preprocessor_kwargs = {"size": env.observation_space, "device": device}
cfg.value_preprocessor = RunningStandardScaler
cfg.value_preprocessor_kwargs = {"size": 1, "device": device}
cfg.experiment.checkpoint_interval = "auto"

agent = SAC(
    models=models,
    memory=memory,
    observation_space=env.observation_space,
    action_space=env.action_space,
    device=device,
    cfg=cfg,
)



[skrl:INFO] Environment wrapper: Gymnasium
INFO:skrl:Environment wrapper: Gymnasium
[skrl:WARNING] Failed to check for a vectorized environment: module 'gymnasium.experimental' has no attribute 'vector'


In [7]:
from skrl.trainers.torch import SequentialTrainer

trainer = SequentialTrainer(cfg=cfg, env=env, agents=agent)
trainer.train()


 69%|██████▊   | 34311/50000 [53:46<25:46, 10.14it/s]  

: 